In [ ]:
!pip install -q -U langgraph langchain langchain-community langchain-ollama gradio

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

print("Ollama server started")

Ollama server started


In [ ]:
!ollama pull qwen3:8b

In [ ]:
!ollama list

NAME        ID              SIZE      MODIFIED               
qwen3:8b    500a1f067a9f    5.2 GB    Less than a second ago    


In [ ]:
from typing import TypedDict, List
from langchain_core.messages import BaseMessage

class RecipeState(TypedDict):
    messages: List[BaseMessage]
    recipe_name: str
    servings: int
    ingredients: List[str]
    steps: List[str]
    current_step: int
    completed_steps: List[int]
    intent: str

    # Timer state
    timer_seconds: int
    timer_started_at: float

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0.2
)

In [ ]:
response = llm.invoke(
    "Give me one sentence explaining how to cook rice."
)

print(response.content)

To cook rice, rinse the grains, add them to water in a 1:2 ratio, and simmer until the water is absorbed, then let it rest covered.


In [ ]:
import json

def recipe_agent(state: RecipeState):

    prompt = f"""
You are an AI cooking assistant.

The user wants help with cooking.

User message:
{state['user_message']}

Current recipe:
{state['recipe_name']}

Servings:
{state['servings']}

Ingredients:
{state['ingredients']}

Cooking steps:
{state['steps']}

Current step:
{state['current_step']}

Completed steps:
{state['completed_steps']}

Give a helpful cooking instruction.

Rules:
1. Focus on the current cooking step.
2. Do not skip steps unless the user asks.
3. Explain cooking terminology when necessary.
4. If the user asks a question, answer it clearly.
5. If the user says they completed the step, move naturally toward the next step.
"""

    result = llm.invoke(prompt)

    state["response"] = result.content

    return state

In [ ]:
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(RecipeState)

graph_builder.add_node(
    "recipe_agent",
    recipe_agent
)

graph_builder.set_entry_point("recipe_agent")

graph_builder.add_edge(
    "recipe_agent",
    END
)

recipe_graph = graph_builder.compile()

In [ ]:
initial_state = {
    "user_message": "How do I make chicken curry?",
    "recipe_name": "Chicken Curry",
    "servings": 4,
    "ingredients": [
        "500g chicken",
        "2 onions",
        "3 tomatoes",
        "2 tablespoons oil",
        "1 teaspoon turmeric",
        "1 teaspoon chilli powder"
    ],
    "steps": [
        "Heat oil in a large pan.",
        "Add onions and cook until golden brown.",
        "Add tomatoes and spices.",
        "Add chicken and cook thoroughly.",
        "Add water and simmer.",
        "Check seasoning.",
        "Serve hot."
    ],
    "current_step": 0,
    "completed_steps": [],
    "response": ""
}

result = recipe_graph.invoke(initial_state)

print(result["response"])

**Step 1: Heat the oil**  
Start by adding 2 tablespoons of oil to a large pan or skillet. Heat it over medium heat until it shimmers but doesn’t smoke. This ensures the onions will cook evenly without burning.  

**Tip:** If using a non-stick pan, make sure it’s preheated properly. Once the oil is hot, proceed to the next step!  

Would you like clarification on any part of this step?


In [ ]:
import gradio as gr

In [ ]:
def chat_with_agent(message, history):

    state = {
        "user_message": message,
        "recipe_name": "Chicken Curry",
        "servings": 4,
        "ingredients": [
            "500g chicken",
            "2 onions",
            "3 tomatoes",
            "2 tablespoons oil",
            "1 teaspoon turmeric",
            "1 teaspoon chilli powder"
        ],
        "steps": [
            "Heat oil in a large pan.",
            "Add onions and cook until golden brown.",
            "Add tomatoes and spices.",
            "Add chicken and cook thoroughly.",
            "Add water and simmer.",
            "Check seasoning.",
            "Serve hot."
        ],
        "current_step": 0,
        "completed_steps": [],
        "response": ""
    }

    result = recipe_graph.invoke(state)

    return result["response"]

In [ ]:
demo = gr.ChatInterface(
    fn=chat_with_agent,
    title="🍳 AI Recipe Chef",
    description="Your step-by-step AI cooking assistant",
    textbox=gr.Textbox(
        placeholder="Ask me anything about cooking...",
        container=True
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://664c8f614558e07f11.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from typing import TypedDict, List, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END


class RecipeState(TypedDict):
    messages: List[BaseMessage]

    recipe_name: str
    servings: int
    ingredients: List[str]
    steps: List[str]

    current_step: int
    completed_steps: List[int]

    intent: str

In [ ]:
def detect_intent(state: RecipeState):

    user_message = state["messages"][-1].content

    prompt = f"""
Classify the user's message into exactly ONE category.

Categories:

NEXT_STEP
QUESTION
REPEAT
START
OTHER

User message:
{user_message}

Return ONLY the category name.
"""

    response = llm.invoke(prompt)

    intent = response.content.strip().upper()

    valid_intents = {
        "NEXT_STEP",
        "QUESTION",
        "REPEAT",
        "START",
        "OTHER"
    }

    if intent not in valid_intents:
        intent = "OTHER"

    return {
        "intent": intent
    }

In [ ]:
def advance_step(state: RecipeState):

    current = state["current_step"]

    completed = list(state["completed_steps"])

    if current not in completed:
        completed.append(current)

    next_step = current + 1

    if next_step >= len(state["steps"]):
        next_step = len(state["steps"])

    return {
        "current_step": next_step,
        "completed_steps": completed
    }

In [ ]:
def cooking_response(state: RecipeState):

    current = state["current_step"]

    if current >= len(state["steps"]):

        response_text = """
🎉 Congratulations! You've completed the recipe.

Your cooking is finished. Enjoy your meal! 🍽️
"""

    else:

        step_number = current + 1
        instruction = state["steps"][current]

        prompt = f"""
You are an AI cooking assistant.

Recipe:
{state["recipe_name"]}

Servings:
{state["servings"]}

The user has completed the previous step.

Now provide the next cooking instruction.

Step {step_number}:
{instruction}

Rules:
- Be concise.
- Explain what the user should do.
- Mention important cooking cues such as colour, texture or approximate time when useful.
- Do not list the entire recipe.
"""

        response = llm.invoke(prompt)

        response_text = response.content

    return {
        "messages": state["messages"] + [
            AIMessage(content=response_text)
        ]
    }

In [ ]:
def answer_question(state: RecipeState):

    current = state["current_step"]

    current_instruction = (
        state["steps"][current]
        if current < len(state["steps"])
        else "The recipe is complete."
    )

    prompt = f"""
You are an AI cooking assistant.

Recipe:
{state["recipe_name"]}

Current step:
{current + 1}

Current instruction:
{current_instruction}

User question:
{state["messages"][-1].content}

Answer the user's question clearly and practically.

Do not restart the recipe.
Do not change the current cooking step.
"""

    response = llm.invoke(prompt)

    return {
        "messages": state["messages"] + [
            AIMessage(content=response.content)
        ]
    }

In [ ]:
def repeat_step(state: RecipeState):

    current = state["current_step"]

    if current >= len(state["steps"]):

        text = "The recipe is already complete."

    else:

        text = f"""
🔄 Let's repeat the current step.

**Step {current + 1}:**

{state["steps"][current]}
"""

    return {
        "messages": state["messages"] + [
            AIMessage(content=text)
        ]
    }

In [ ]:
def start_recipe(state: RecipeState):

    current = state["current_step"]

    instruction = state["steps"][current]

    text = f"""
👨‍🍳 Let's start cooking **{state["recipe_name"]}**!

### Step {current + 1}

{instruction}

When you're finished, just say **"done"** and I'll guide you to the next step.
"""

    return {
        "messages": state["messages"] + [
            AIMessage(content=text)
        ]
    }

In [ ]:
def route_intent(state: RecipeState):

    intent = state["intent"]

    if intent == "NEXT_STEP":
        return "advance"

    elif intent == "QUESTION":
        return "question"

    elif intent == "REPEAT":
        return "repeat"

    elif intent == "START":
        return "start"

    else:
        return "question"

In [ ]:
graph_builder = StateGraph(RecipeState)

graph_builder.add_node(
    "detect_intent",
    detect_intent
)

graph_builder.add_node(
    "advance_step",
    advance_step
)

graph_builder.add_node(
    "cooking_response",
    cooking_response
)

graph_builder.add_node(
    "answer_question",
    answer_question
)

graph_builder.add_node(
    "repeat_step",
    repeat_step
)

graph_builder.add_node(
    "start_recipe",
    start_recipe
)

In [ ]:
graph_builder.set_entry_point("detect_intent")

In [ ]:
graph_builder.add_conditional_edges(
    "detect_intent",
    route_intent,
    {
        "advance": "advance_step",
        "question": "answer_question",
        "repeat": "repeat_step",
        "start": "start_recipe"
    }
)

ValueError: Branch with name `route_intent` already exists for node `detect_intent`

In [ ]:
graph_builder.add_edge(
    "advance_step",
    "cooking_response"
)

In [ ]:
graph_builder.add_edge(
    "cooking_response",
    END
)

graph_builder.add_edge(
    "answer_question",
    END
)

graph_builder.add_edge(
    "repeat_step",
    END
)

graph_builder.add_edge(
    "start_recipe",
    END
)

In [ ]:
recipe_agent = graph_builder.compile()

print("✅ Recipe Agent compiled successfully!")

✅ Recipe Agent compiled successfully!


In [ ]:
recipe = {
    "recipe_name": "Chicken Curry",
    "servings": 4,

    "ingredients": [
        "500g chicken",
        "2 onions, chopped",
        "3 tomatoes, chopped",
        "2 tablespoons cooking oil",
        "1 teaspoon turmeric",
        "1 teaspoon chilli powder",
        "1 teaspoon garam masala",
        "200ml water",
        "Salt to taste"
    ],

    "steps": [
        "Heat 2 tablespoons of oil in a large pan over medium heat.",
        "Add the chopped onions and cook until they become golden brown.",
        "Add tomatoes, turmeric, chilli powder and salt. Cook until the tomatoes soften.",
        "Add the chicken and mix well with the spices.",
        "Add 200ml water, cover the pan and cook the chicken thoroughly.",
        "Add garam masala and simmer for a few minutes.",
        "Check the seasoning and serve the chicken curry hot."
    ]
}

print("✅ Recipe loaded:", recipe["recipe_name"])
print("🍽️ Servings:", recipe["servings"])
print("📋 Steps:", len(recipe["steps"]))

✅ Recipe loaded: Chicken Curry
🍽️ Servings: 4
📋 Steps: 7


In [ ]:
state = {
    "messages": [
        HumanMessage(content="Start cooking")
    ],

    "recipe_name": recipe["recipe_name"],
    "servings": recipe["servings"],
    "ingredients": recipe["ingredients"],
    "steps": recipe["steps"],

    "current_step": 0,
    "completed_steps": [],
    "intent": ""
}

In [ ]:
result = recipe_agent.invoke(state)

print(result["messages"][-1].content)
print("Current step:", result["current_step"])


👨‍🍳 Let's start cooking **Chicken Curry**!

### Step 1

Heat 2 tablespoons of oil in a large pan over medium heat.

When you're finished, just say **"done"** and I'll guide you to the next step.

Current step: 0


In [ ]:
state = result

state["messages"].append(
    HumanMessage(content="Done")
)

result = recipe_agent.invoke(state)

print(result["messages"][-1].content)
print("Current step:", result["current_step"])

Add minced garlic and grated ginger, stirring frequently until fragrant (about 1–2 minutes) and the mixture turns a light golden hue.
Current step: 1


In [ ]:
state = result

state["messages"].append(
    HumanMessage(
        content="Can I use olive oil instead of normal cooking oil?"
    )
)

result = recipe_agent.invoke(state)

print(result["messages"][-1].content)
print("Current step:", result["current_step"])

Yes, you can use olive oil instead of normal cooking oil for this step. Olive oil works well for sautéing onions and will impart a mild, fruity flavor that complements the curry. Use a neutral olive oil (like light or regular olive oil) for a milder taste, or extra virgin olive oil if you prefer a more robust flavor. Ensure the heat is medium to medium-high to avoid burning, as olive oil has a lower smoke point than some other oils. Proceed as instructed! 🥗
Current step: 1


In [ ]:
conversation_state = {
    "messages": [],
    "recipe_name": recipe["recipe_name"],
    "servings": recipe["servings"],
    "ingredients": recipe["ingredients"],
    "steps": recipe["steps"],
    "current_step": 0,
    "completed_steps": [],
    "intent": ""
}

print("✅ Cooking session created")
print("Recipe:", conversation_state["recipe_name"])
print("Current step:", conversation_state["current_step"] + 1)

✅ Cooking session created
Recipe: Chicken Curry
Current step: 1


In [ ]:
def process_message(user_message):

    conversation_state["messages"].append(
        HumanMessage(content=user_message)
    )

    result = recipe_agent.invoke(conversation_state)

    # Update persistent state
    conversation_state.update(result)

    return result["messages"][-1].content

In [ ]:
print(process_message("Start cooking"))


👨‍🍳 Let's start cooking **Chicken Curry**!

### Step 1

Heat 2 tablespoons of oil in a large pan over medium heat.

When you're finished, just say **"done"** and I'll guide you to the next step.



In [ ]:
print(process_message("Done"))

Add minced garlic and grated ginger, stirring frequently until fragrant (about 1–2 minutes). The mixture should smell aromatic and just begin to turn a light golden hue.


In [ ]:
print("Current step:", conversation_state["current_step"] + 1)
print("Completed:", conversation_state["completed_steps"])

Current step: 2
Completed: [0]


In [ ]:
!pip install -q -U gradio

In [ ]:
import gradio as gr

In [ ]:
def gradio_chat(message, history):
    if not message or not message.strip():
        return "", history

    response = process_message(message)

    if history is None:
        history = []

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": response
    })

    return "", history

In [ ]:
import subprocess
import time
import requests

# Check whether Ollama is already running
try:
    r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
    print("✅ Ollama is already running")
except Exception:
    print("⚠️ Ollama is not running. Starting it...")

    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(5)

    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
        print("✅ Ollama server started")
    except Exception as e:
        print("❌ Ollama failed to start:", e)

⚠️ Ollama is not running. Starting it...
✅ Ollama server started


In [ ]:
import gradio as gr

with gr.Blocks(title="AI Recipe Chef") as demo:

    gr.Markdown("""
    # 🍳 AI Recipe Chef
    ### Your step-by-step AI cooking assistant
    """)

    chatbot = gr.Chatbot(
        label="Cooking Assistant",
        height=500
    )

    with gr.Row():

        message_box = gr.Textbox(
            placeholder="Ask something like: Start cooking",
            label="Message",
            scale=4
        )

        send_button = gr.Button(
            "Send",
            variant="primary",
            scale=1
        )

    with gr.Row():

        start_button = gr.Button("🍳 Start Cooking")
        done_button = gr.Button("✅ Done")
        repeat_button = gr.Button("🔄 Repeat Step")

    # Normal message
    send_button.click(
        fn=gradio_chat,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot]
    )

    # Press Enter
    message_box.submit(
        fn=gradio_chat,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot]
    )

    # Start cooking
    start_button.click(
        fn=lambda history: gradio_chat(
            "Start cooking",
            history
        ),
        inputs=[chatbot],
        outputs=[message_box, chatbot]
    )

    # Done
    done_button.click(
        fn=lambda history: gradio_chat(
            "Done",
            history
        ),
        inputs=[chatbot],
        outputs=[message_box, chatbot]
    )

    # Repeat
    repeat_button.click(
        fn=lambda history: gradio_chat(
            "Repeat the current step",
            history
        ),
        inputs=[chatbot],
        outputs=[message_box, chatbot]
    )


demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0467f3de4b9a32e2ad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://0467f3de4b9a32e2ad.gradio.live


In [ ]:
import json
import re

def generate_recipe(state: RecipeState):
    user_message = state["messages"][-1].content

    prompt = f"""
You are an expert cooking recipe generator.

Create a practical recipe based on the user's request.

USER REQUEST:
{user_message}

Return ONLY valid JSON.
Do not use markdown.
Do not add explanations outside the JSON.

Use exactly this structure:

{{
    "recipe_name": "Recipe name",
    "servings": 4,
    "ingredients": [
        "ingredient 1",
        "ingredient 2"
    ],
    "steps": [
        "step 1",
        "step 2",
        "step 3"
    ]
}}

Requirements:
- Use the requested number of servings if provided.
- If no serving size is provided, use 4.
- Give realistic ingredient quantities.
- Give clear sequential cooking steps.
- Include enough steps for an interactive cooking session.
"""

    response = llm.invoke(prompt)

    text = response.content.strip()

    # Remove accidental markdown fences
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)

    recipe_data = json.loads(text)

    return {
        "recipe_name": recipe_data["recipe_name"],
        "servings": recipe_data["servings"],
        "ingredients": recipe_data["ingredients"],
        "steps": recipe_data["steps"],
        "current_step": 0,
        "completed_steps": []
    }

In [ ]:
test_state = {
    "messages": [
        HumanMessage(
            content="I want chicken biryani for 5 people"
        )
    ],
    "recipe_name": "",
    "servings": 0,
    "ingredients": [],
    "steps": [],
    "current_step": 0,
    "completed_steps": [],
    "intent": "START"
}

result = generate_recipe(test_state)

print("Recipe:", result["recipe_name"])
print("Servings:", result["servings"])

print("\nIngredients:")
for ingredient in result["ingredients"]:
    print("-", ingredient)

print("\nSteps:")
for i, step in enumerate(result["steps"], 1):
    print(f"{i}. {step}")

Recipe: Chicken Biryani
Servings: 5

Ingredients:
- 500g chicken thighs
- 1.5 cups basmati rice
- 2 medium onions
- 2 medium tomatoes
- 1 tbsp ginger-garlic paste
- 2 green chilies
- 1 small cardamom
- 1 small cinnamon stick
- 2 bay leaves
- 1 tsp cumin seeds
- Salt to taste
- 1 tsp garam masala
- 1/2 tsp turmeric
- 1 tsp chili powder
- 2 tbsp yogurt
- 2 tbsp oil

Steps:
1. Marinate chicken with yogurt, ginger-garlic paste, salt, chili powder, and 1 tsp garam masala for 30 minutes.
2. Wash basmati rice, soak in water for 30 minutes, drain and set aside.
3. Heat oil in a large pot, sauté onions until golden, add tomatoes and cook until softened.
4. Add cumin seeds, cardamom, cinnamon, bay leaves, and turmeric. Cook for 2 minutes.
5. Add marinated chicken and cook until browned. Stir in green chilies and garam masala.
6. Layer cooked rice over the chicken mixture, pour saffron water (1/4 cup warm water + 1/4 tsp saffron), and mix gently.
7. Cover and let rest for 10 minutes. Steam on low

In [ ]:
from langgraph.graph import StateGraph, END

# Create a fresh graph builder
graph_builder = StateGraph(RecipeState)

# Add nodes
graph_builder.add_node("detect_intent", detect_intent)
graph_builder.add_node("generate_recipe", generate_recipe)
graph_builder.add_node("advance_step", advance_step)
graph_builder.add_node("cooking_response", cooking_response)
graph_builder.add_node("answer_question", answer_question)
graph_builder.add_node("repeat_step", repeat_step)
graph_builder.add_node("start_recipe", start_recipe)

# Entry point
graph_builder.set_entry_point("detect_intent")

# Intent routing
graph_builder.add_conditional_edges(
    "detect_intent",
    route_intent,
    {
        "advance": "advance_step",
        "question": "answer_question",
        "repeat": "repeat_step",
        "start": "generate_recipe"
    }
)

# Normal edges
graph_builder.add_edge("advance_step", "cooking_response")
graph_builder.add_edge("cooking_response", END)

graph_builder.add_edge("answer_question", END)

graph_builder.add_edge("repeat_step", END)

# New dynamic recipe flow
graph_builder.add_edge("generate_recipe", "start_recipe")
graph_builder.add_edge("start_recipe", END)

# Compile
recipe_agent = graph_builder.compile()

print("✅ Dynamic Recipe Agent compiled successfully")

✅ Dynamic Recipe Agent compiled successfully


In [ ]:
conversation_state = {
    "messages": [],
    "recipe_name": "",
    "servings": 0,
    "ingredients": [],
    "steps": [],
    "current_step": 0,
    "completed_steps": [],
    "intent": ""
}

print("✅ Conversation state reset")

✅ Conversation state reset


In [ ]:
print(process_message("I want chicken biryani for 5 people"))


👨‍🍳 Let's start cooking **Chicken Biryani**!

### Step 1

Soak basmati rice for 30 minutes, drain and set aside.

When you're finished, just say **"done"** and I'll guide you to the next step.



In [ ]:
def scale_recipe(state: RecipeState):
    user_message = state["messages"][-1].content

    prompt = f"""
You are a cooking assistant.

The current recipe is:

Recipe: {state["recipe_name"]}
Current servings: {state["servings"]}

Ingredients:
{state["ingredients"]}

The user wants:
{user_message}

Determine the new number of servings and scale the ingredient quantities accordingly.

Return ONLY valid JSON:

{{
    "servings": 8,
    "ingredients": [
        "ingredient with scaled quantity",
        "ingredient with scaled quantity"
    ]
}}

Rules:
- Preserve the same recipe.
- Scale quantities proportionally.
- Do not change the cooking steps.
- Return only JSON.
"""

    response = llm.invoke(prompt)

    import json
    import re

    text = response.content.strip()
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)

    data = json.loads(text)

    return {
        "servings": data["servings"],
        "ingredients": data["ingredients"]
    }

In [ ]:
def detect_intent(state: RecipeState):
    user_message = state["messages"][-1].content

    prompt = f"""
Classify the user's message into exactly ONE category.

Categories:

NEXT_STEP
QUESTION
REPEAT
START
SCALE
OTHER

User message:
{user_message}

Return ONLY the category name.
"""

    response = llm.invoke(prompt)

    intent = response.content.strip().upper()

    valid_intents = {
        "NEXT_STEP",
        "QUESTION",
        "REPEAT",
        "START",
        "SCALE",
        "OTHER"
    }

    if intent not in valid_intents:
        intent = "OTHER"

    return {"intent": intent}

In [ ]:
def route_intent(state: RecipeState):
    intent = state["intent"]

    if intent == "NEXT_STEP":
        return "advance"

    elif intent == "QUESTION":
        return "question"

    elif intent == "REPEAT":
        return "repeat"

    elif intent == "START":
        return "start"

    elif intent == "SCALE":
        return "scale"

    else:
        return "question"

In [ ]:
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(RecipeState)

graph_builder.add_node("detect_intent", detect_intent)
graph_builder.add_node("generate_recipe", generate_recipe)
graph_builder.add_node("scale_recipe", scale_recipe)
graph_builder.add_node("advance_step", advance_step)
graph_builder.add_node("cooking_response", cooking_response)
graph_builder.add_node("answer_question", answer_question)
graph_builder.add_node("repeat_step", repeat_step)
graph_builder.add_node("start_recipe", start_recipe)

graph_builder.set_entry_point("detect_intent")

graph_builder.add_conditional_edges(
    "detect_intent",
    route_intent,
    {
        "advance": "advance_step",
        "question": "answer_question",
        "repeat": "repeat_step",
        "start": "generate_recipe",
        "scale": "scale_recipe"
    }
)

graph_builder.add_edge("generate_recipe", "start_recipe")
graph_builder.add_edge("start_recipe", END)

graph_builder.add_edge("scale_recipe", END)

graph_builder.add_edge("advance_step", "cooking_response")
graph_builder.add_edge("cooking_response", END)

graph_builder.add_edge("answer_question", END)
graph_builder.add_edge("repeat_step", END)

recipe_agent = graph_builder.compile()

print("✅ Recipe Agent updated successfully")

✅ Recipe Agent updated successfully


In [ ]:
conversation_state = {
    "messages": [],
    "recipe_name": "",
    "servings": 0,
    "ingredients": [],
    "steps": [],
    "current_step": 0,
    "completed_steps": [],
    "intent": ""
}

In [ ]:
print(process_message("I want chicken biryani for 5 people"))

I want chicken biryani for 5 people


In [ ]:
print(process_message("Actually, make it for 8 people"))

Actually, make it for 8 people


In [ ]:
from langchain_core.messages import AIMessage
import json
import re

def substitute_ingredient(state: RecipeState):
    user_message = state["messages"][-1].content

    prompt = f"""
You are an expert cooking assistant.

Current recipe:
{state["recipe_name"]}

Servings:
{state["servings"]}

Current ingredients:
{state["ingredients"]}

Current cooking steps:
{state["steps"]}

User request:
{user_message}

Determine the ingredient substitution requested by the user.

Return ONLY valid JSON:

{{
    "ingredients": [
        "updated ingredient 1",
        "updated ingredient 2"
    ],
    "steps": [
        "updated step 1",
        "updated step 2"
    ],
    "message": "Short explanation of the substitution"
}}

Rules:
- Keep the same recipe.
- Replace only the requested ingredient.
- Adjust the quantity if necessary.
- Modify cooking steps if the substitute requires different cooking instructions.
- Keep all unrelated ingredients and steps unchanged.
- Return only JSON.
"""

    response = llm.invoke(prompt)

    text = response.content.strip()

    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)

    data = json.loads(text)

    return {
        "ingredients": data["ingredients"],
        "steps": data["steps"],
        "messages": [
            AIMessage(content=data["message"])
        ]
    }

In [ ]:
def detect_intent(state: RecipeState):
    user_message = state["messages"][-1].content

    prompt = f"""
Classify the user's message into exactly ONE category.

Categories:

NEXT_STEP
QUESTION
REPEAT
START
SCALE
SUBSTITUTE
TIMER
OTHER

User message:
{user_message}

Return ONLY the category name.
"""

    response = llm.invoke(prompt)

    intent = response.content.strip().upper()

    valid_intents = {
        "NEXT_STEP",
        "QUESTION",
        "REPEAT",
        "START",
        "SCALE",
        "SUBSTITUTE",
        "TIMER",
        "OTHER"
    }

    if intent not in valid_intents:
        intent = "OTHER"

    return {"intent": intent}

In [ ]:
import re
from langchain_core.messages import AIMessage

def timer_agent(state: RecipeState):
    user_message = state["messages"][-1].content

    prompt = f"""
You are a cooking timer assistant.

User message:
{user_message}

Extract the requested timer duration.

Return ONLY valid JSON:

{{
    "duration_seconds": 600,
    "duration_text": "10 minutes",
    "message": "Timer set for 10 minutes."
}}

Rules:
- Convert minutes to seconds.
- Convert hours to seconds.
- If the user says "half an hour", use 1800 seconds.
- If no duration can be identified, return duration_seconds as 0.
- Return ONLY JSON.
"""

    response = llm.invoke(prompt)

    text = response.content.strip()

    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)

    import json
    data = json.loads(text)

    return {
        "messages": [
            AIMessage(content=data["message"])
        ]
    }

In [ ]:
def route_intent(state: RecipeState):
    intent = state["intent"]

    if intent == "NEXT_STEP":
        return "advance"

    elif intent == "QUESTION":
        return "question"

    elif intent == "REPEAT":
        return "repeat"

    elif intent == "START":
        return "start"

    elif intent == "SCALE":
        return "scale"

    elif intent == "SUBSTITUTE":
        return "substitute"

    elif intent == "TIMER":
        return "timer"

    else:
        return "question"

In [ ]:
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(RecipeState)

# Nodes
graph_builder.add_node("detect_intent", detect_intent)
graph_builder.add_node("generate_recipe", generate_recipe)
graph_builder.add_node("scale_recipe", scale_recipe)
graph_builder.add_node("substitute_ingredient", substitute_ingredient)
graph_builder.add_node("timer_agent", timer_agent)

graph_builder.add_node("advance_step", advance_step)
graph_builder.add_node("cooking_response", cooking_response)
graph_builder.add_node("answer_question", answer_question)
graph_builder.add_node("repeat_step", repeat_step)
graph_builder.add_node("start_recipe", start_recipe)

# Entry
graph_builder.set_entry_point("detect_intent")

# Routing
graph_builder.add_conditional_edges(
    "detect_intent",
    route_intent,
    {
        "advance": "advance_step",
        "question": "answer_question",
        "repeat": "repeat_step",
        "start": "generate_recipe",
        "scale": "scale_recipe",
        "substitute": "substitute_ingredient",
        "timer": "timer_agent"
    }
)

# Recipe generation
graph_builder.add_edge("generate_recipe", "start_recipe")
graph_builder.add_edge("start_recipe", END)

# Other agents
graph_builder.add_edge("scale_recipe", END)
graph_builder.add_edge("substitute_ingredient", END)
graph_builder.add_edge("timer_agent", END)

# Cooking
graph_builder.add_edge("advance_step", "cooking_response")
graph_builder.add_edge("cooking_response", END)

graph_builder.add_edge("answer_question", END)
graph_builder.add_edge("repeat_step", END)

recipe_agent = graph_builder.compile()

print("✅ Recipe Agent with Timer compiled successfully")

✅ Recipe Agent with Timer compiled successfully


In [ ]:
conversation_state = {
    "messages": [],
    "recipe_name": "",
    "servings": 0,
    "ingredients": [],
    "steps": [],
    "current_step": 0,
    "completed_steps": [],
    "intent": "",

    "timer_seconds": 0,
    "timer_started_at": 0.0
}

In [ ]:
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"

def ensure_ollama():
    # Check whether Ollama is already running
    try:
        r = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=3
        )

        if r.status_code == 200:
            print("✅ Ollama is already running")
            return

    except Exception:
        pass

    print("🚀 Starting Ollama...")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # Wait for Ollama
    for i in range(20):
        time.sleep(1)

        try:
            r = requests.get(
                f"{OLLAMA_URL}/api/tags",
                timeout=2
            )

            if r.status_code == 200:
                print(f"✅ Ollama started successfully ({i+1}s)")
                return

        except Exception:
            pass

    raise RuntimeError(
        "❌ Ollama failed to start. "
        "Restart the Colab runtime and run the installation cell again."
    )

ensure_ollama()

🚀 Starting Ollama...
✅ Ollama started successfully (2s)


In [ ]:
print(process_message("I want chicken curry for 4 people"))

I want chicken curry for 4 people


In [ ]:
print(process_message("I don't have chicken, can I use paneer instead?"))

Paneer can replace chicken in most recipes. Adjust cooking time to prevent sogginess and maintain texture.


In [ ]:
print(process_message("Set a timer for 1 hour"))

Timer set for 1 hour.


In [ ]:
import time

def timer_display():
    duration = conversation_state.get("timer_seconds", 0)
    started = conversation_state.get("timer_started_at", 0)

    if duration <= 0 or started <= 0:
        return "⏱️ No active timer"

    elapsed = int(time.time() - started)
    remaining = max(0, duration - elapsed)

    if remaining == 0:
        return "🔔 **Timer finished!**"

    minutes = remaining // 60
    seconds = remaining % 60

    return f"⏱️ **{minutes:02d}:{seconds:02d} remaining**"

In [ ]:
def refresh_timer():
    return timer_display()

In [ ]:
import gradio as gr

with gr.Blocks(title="AI Recipe Chef") as demo:

    gr.Markdown("""
    # 🍳 AI Recipe Chef
    ### Your step-by-step AI cooking assistant
    """)

    chatbot = gr.Chatbot(
        label="Cooking Assistant",
        height=500
    )

    # Timer display
    timer_output = gr.Markdown(
        "⏱️ **No active timer**"
    )

    with gr.Row():

        message_box = gr.Textbox(
            placeholder="Ask something like: Start cooking",
            label="Message",
            scale=4
        )

        send_button = gr.Button(
            "Send",
            variant="primary",
            scale=1
        )

    with gr.Row():

        start_button = gr.Button("🍳 Start Cooking")

        done_button = gr.Button("✅ Done")

        repeat_button = gr.Button("🔄 Repeat Step")

    # -------------------------
    # Chat
    # -------------------------

    send_button.click(
        fn=gradio_chat,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot]
    )

    message_box.submit(
        fn=gradio_chat,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot]
    )

    # -------------------------
    # Buttons
    # -------------------------

    start_button.click(
        fn=lambda history: gradio_chat(
            "Start cooking",
            history
        ),
        inputs=[chatbot],
        outputs=[message_box, chatbot]
    )

    done_button.click(
        fn=lambda history: gradio_chat(
            "Done",
            history
        ),
        inputs=[chatbot],
        outputs=[message_box, chatbot]
    )

    repeat_button.click(
        fn=lambda history: gradio_chat(
            "Repeat the current step",
            history
        ),
        inputs=[chatbot],
        outputs=[message_box, chatbot]
    )

    # -------------------------
    # Timer
    # -------------------------

    timer = gr.Timer(1)

    timer.tick(
        fn=refresh_timer,
        inputs=[],
        outputs=timer_output
    )


demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://69b14bc53ad73876c0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
